# GPU run — MagnaTagATune + GTZAN

Runs the whole experimental suite on a free Colab T4. CPU-only training of the
BERT-based tasks takes 6–8 hours; this takes roughly 40–70 minutes.

**Before you start:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

What this does:

1. clones the repo and installs the pinned dependencies
2. downloads MagnaTagATune (~2.8 GB) *inside* Colab — far faster than uploading
3. Tasks 1, 3 (4-arm ablation) and 4 + baselines on **MagnaTagATune** (the spec's
   corpus for Tasks 1 and 3)
4. Task 2 + GAT + a segment/chord/hybrid graph comparison on **GTZAN** (the
   spec's corpus for Task 2)
5. zips `results_*/` for download

Each corpus writes to its own results directory — mixing corpora in one metrics
file is a hard error, by design.

In [1]:
!nvidia-smi -L
import torch, os
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then re-run."

GPU 0: Tesla T4 (UUID: GPU-bc720cad-5de0-a640-1be3-cbef0f9e3f64)
torch: 2.11.0+cu128 | cuda: True


## 1. Get the code

The repo URL is already filled in. **If the repo is private**, paste a GitHub
personal access token into `TOKEN` in the next cell — Colab cannot clone a
private repo without one. Leave `TOKEN` empty if the repo is public.

(Token: GitHub → Settings → Developer settings → Personal access tokens →
Tokens (classic) → Generate new token → tick `repo`.)

In [2]:
REPO = "sharminahmednova/Supervised-Neural-Network-Project-GNN-Based-BERT-for-Understanding-Context-from-Music"

# Only needed if the repo is PRIVATE: GitHub > Settings > Developer settings >
# Personal access tokens > Tokens (classic), tick `repo`. Leave "" if public.
TOKEN = ""

REPO_URL = (f"https://{TOKEN}@github.com/{REPO}.git" if TOKEN
            else f"https://github.com/{REPO}.git")

# Clone into a FIXED directory. `git clone` otherwise names the folder after
# the repo, so every path below would depend on how long the repo name is.
WORKDIR = "/content/project"

import os, subprocess
from pathlib import Path

if (Path(WORKDIR) / ".git").exists():
    subprocess.run(["git", "-C", WORKDIR, "pull", "-q"], check=False)
    print("updated existing clone")
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, WORKDIR], check=True)
    print("cloned")

os.chdir(WORKDIR)
print("cwd:", Path.cwd())
subprocess.run(["ls"])

cloned
cwd: /content/project


CompletedProcess(args=['ls'], returncode=0)

In [3]:
# ALTERNATIVE to the clone above, if the clone fails (private repo, no token).
# Zip your local project folder, upload it here, and carry on.
# import os, glob, subprocess
# from google.colab import files
# from pathlib import Path
# files.upload()                                  # pick your .zip
# zip_name = sorted(glob.glob("*.zip"))[0]
# subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "/content/unzipped"])
# # the zip may or may not contain a top-level folder -- find the real root
# root = next(p.parent for p in Path("/content/unzipped").rglob("config.yaml"))
# os.chdir(root); print("cwd:", Path.cwd())

## 2. Dependencies

In [4]:
# Colab already ships a CUDA torch; install the rest pinned.
!pip install -q torch-geometric==2.8.0.post1 transformers==5.17.0 librosa==0.11.0 soundfile==0.14.0 pyyaml==6.0.3
!python tools/check_env.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 119.6 MB/s eta 0:00:00
interpreter : /usr/bin/python3
python      : 3.13.15
platform    : Linux 6.6.122+
repo        : /content/project

  ~  torch                2.11.0+cu128  (pinned 2.10.0)
  OK torch-geometric      2.8.0.post1
  OK transformers         5.17.0
  OK librosa              0.11.0
  OK soundfile            0.14.0
  ~  numpy                2.1.3  (pinned 2.2.6)
  ~  scipy                1.16.3  (pinned 1.15.3)
  ~  pandas               2.2.3  (pinned 2.3.3)
  ~  scikit-learn         1.6.1  (pinned 1.7.2)
  ~  matplotlib           3.10.0  (pinned 3.10.6)
  OK pyyaml               6.0.3
  ~  tqdm                 4.67.3  (pinned 4.70.1)
  -- yt-dlp               not installed (optional)

device      : cuda -- Tesla T4

Versions differ from the pins that produced th

## 3. Fetch the corpora

MagnaTagATune is ~2.8 GB as a 3-part split zip; GTZAN is ~1.1 GB. Colab's link makes this minutes rather than hours.

In [5]:
!python tools/fetch_data.py magnatagatune --extract
!python tools/fetch_data.py gtzan --extract
!du -sh data/raw/*


== magnatagatune ==  ~2.8 GB
  25,877 clips / 29 s, 188 human tags. Audio is a 3-part split zip.
  disk free: 65.1 GB   needed: ~6 GB (parts + joined zip + extracted)
  remote size: 20.52 MB
  20.52 MB / 20.52 MB  100.0%   11.6 MB/s  ETA   0.0 min   
  complete: annotations_final.csv (20.52 MB)
  remote size: 1.02 GB
  1.02 GB / 1.02 GB  100.0%   18.4 MB/s  ETA   0.0 min   
  complete: mp3.zip.001 (1.02 GB)
  remote size: 1.02 GB
  1.02 GB / 1.02 GB  100.0%   18.4 MB/s  ETA   0.0 min   
  complete: mp3.zip.002 (1.02 GB)
  remote size: 736.97 MB
  736.97 MB / 736.97 MB  100.0%   18.5 MB/s  ETA   0.0 min   
  complete: mp3.zip.003 (736.97 MB)
  joining 3 parts -> mp3.zip (2.77 GB)
  extracting mp3.zip -> /content/project/data/raw/magnatagatune
  extracted
  removed mp3.zip
  removed split parts
  25863 mp3 files present (expect ~25,863)

done. Next:
  python src/preprocess.py --set dataset.source=magnatagatune

== gtzan ==  ~1.14 GB
  1,000 clips / 30 s / 10 genres. Smallest real corpus

## 4. MagnaTagATune — Tasks 1, 3, 4 + baselines

`configs/mtat.yaml` sets the honest configuration:

- `top_tags: 50` — the spec's top-50 subset
- `text_tag_fraction: 0.4` — MTAT has no captions, so 40% of the vocabulary
  becomes the "caption" and the **disjoint** remainder is the prediction target.
  The text therefore cannot leak the labels it is scored against. At 0.0 the
  text restates the target and Task 1 scores a meaningless 1.0.
- `segment_seconds: 2.5` — 29 s clips at 5 s windows give a *six node* graph at
  ~0.54 density, where message passing ≈ mean pooling. 2.5 s gives 12 nodes at
  ~0.27 density.

In [6]:
!python src/preprocess.py --config configs/mtat.yaml --export-samples 20
!python -c "import json;m=json.load(open('data/processed_mtat/meta.json'));print({k:v for k,v in m.items() if k!='tag_counts'})"

source=magnatagatune  graph=segment  text=distilbert-base-uncased
config.json: 100% 483/483 [00:00<00:00, 2.68MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 227kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 12.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 27.8MB/s]
  magnatagatune: 21111 records on disk
  subsampled 21111 -> 8000 tracks (stratified by split x genre, seed=42)
    splits: {'val': 579, 'train': 5779, 'test': 1642}
    genres: {'_': 8000}
  featurised 25 tracks...
  featurised 50 tracks...
  featurised 75 tracks...
  featurised 100 tracks...
  featurised 125 tracks...
  featurised 150 tracks...
  featurised 175 tracks...
  featurised 200 tracks...
  featurised 225 tracks...
  featurised 250 tracks...
  featurised 275 tracks...
  featurised 300 tracks...
  featurised 325 tracks...
  featurised 350 tracks...
  featurised 375 tracks...
  featurised 400 tracks...
  featurised 425 tracks...
  featurised 450 tracks...
  featurised 475 tracks...
  featurised 500 t

In [7]:
# GPU: larger batch and a full schedule are affordable here.
!python src/train.py all --config configs/mtat.yaml \
    --set device=cuda --set train.epochs=30 --set train.batch_size=64 \
    --set train.patience=8 --set text.unfreeze_last_n=4


BASELINES: B1 prior/random, B2 mel-CNN, B4 PCA+MLP
  B1 [random]  macro_f1=0.1428  micro_f1=0.1483  auc_pr=0.0839
  B1 [prior ]  macro_f1=0.1153  micro_f1=0.1823  auc_pr=0.0816
[baseline_b2_melcnn] params: 0.10M total, 0.10M trainable | device=cuda
  epoch   1/30  loss=1.0335  val_macro_f1=0.2136  val_micro_f1=0.2520  val_auc_pr=0.2028  (4.9s)  *
  epoch   2/30  loss=0.9406  val_macro_f1=0.2516  val_micro_f1=0.3124  val_auc_pr=0.2274  (3.9s)  *
  epoch   3/30  loss=0.9009  val_macro_f1=0.2557  val_micro_f1=0.3175  val_auc_pr=0.2326  (3.3s)  *
  epoch   4/30  loss=0.8676  val_macro_f1=0.2206  val_micro_f1=0.2236  val_auc_pr=0.2307  (3.2s)
  epoch   5/30  loss=0.8401  val_macro_f1=0.2572  val_micro_f1=0.3206  val_auc_pr=0.2427  (3.7s)  *
  epoch   6/30  loss=0.8267  val_macro_f1=0.2860  val_micro_f1=0.3429  val_auc_pr=0.2479  (3.6s)  *
  epoch   7/30  loss=0.8137  val_macro_f1=0.2238  val_micro_f1=0.2309  val_auc_pr=0.2441  (3.2s)
  epoch   8/30  loss=0.7938  val_macro_f1=0.2378  val_mi

### Task 4 on MagnaTagATune -- needs its own preprocessing

~35% of MTAT records have no text-side tag after the disjoint partition and
share one placeholder string. Identical text ties exactly in retrieval (~61
ties per query, measured), so retrieval must be run on a corpus with those
records dropped. This writes to its own processed/results directories.

In [8]:
!python src/preprocess.py --config configs/mtat.yaml     --set dataset.drop_textless=true     --set paths.processed=data/processed_mtat_r4     --set paths.splits=data/splits_mtat_r4 --export-samples 0
!python src/train.py task4 --config configs/mtat.yaml     --set dataset.drop_textless=true     --set paths.processed=data/processed_mtat_r4     --set paths.splits=data/splits_mtat_r4     --set paths.results=results_mtat_r4     --set device=cuda --set train.epochs=40 --set train.batch_size=64     --set train.patience=10 --set text.unfreeze_last_n=4

source=magnatagatune  graph=segment  text=distilbert-base-uncased
  magnatagatune: 21111 records on disk
  subsampled 21111 -> 8000 tracks (stratified by split x genre, seed=42)
    splits: {'val': 579, 'train': 5779, 'test': 1642}
    genres: {'_': 8000}
  featurised 25 tracks...
  featurised 50 tracks...
  featurised 75 tracks...
  featurised 100 tracks...
  featurised 125 tracks...
  featurised 150 tracks...
  featurised 175 tracks...
  featurised 200 tracks...
  featurised 225 tracks...
  featurised 250 tracks...
  featurised 275 tracks...
  featurised 300 tracks...
  featurised 325 tracks...
  featurised 350 tracks...
  featurised 375 tracks...
  featurised 400 tracks...
  featurised 425 tracks...
  featurised 450 tracks...
  featurised 475 tracks...
  featurised 500 tracks...
  featurised 525 tracks...
  featurised 550 tracks...
  featurised 575 tracks...
  featurised 600 tracks...
  featurised 625 tracks...
  featurised 650 tracks...
  featurised 675 tracks...
  featurised 700 t

### Does cross-attention actually beat early concat?

On the synthetic corpus early concat (0.837) beat cross-attention (0.807), and
cross-attention did not beat `bert_only` (0.809) — but its validation curve was
still rising at the epoch cap, so undertraining is a live alternative
explanation. This cell gives the cross-attention arm twice the schedule; if it
still does not win, the negative result is real and worth reporting.

In [9]:
!python src/train.py task3 --config configs/mtat.yaml --mode cross_attention \
    --set device=cuda --set train.epochs=60 --set train.batch_size=64 \
    --set train.patience=12 --set text.unfreeze_last_n=4 \
    --set paths.results=results_mtat_longsched


TASK 3 (Hard): GNN-BERT fusion [cross_attention]
Loading weights: 100% 100/100 [00:00<00:00, 7693.01it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[task3_fusion_cross_attention] params: 67.37M total, 29.36M trainable | device=cuda
  epoch   1/60  loss=1.0531  val_macro_f1=0.2678  val_micro_f1=0.2956  val_auc_pr=0.2512  (15.2s)  *
  epoch   2/60  loss=0.8793  val_macro_f1=0.2998  val_micro_f1=0.3562  val_auc_pr=0.2955  (15.6s)  *
  epoch   3/60  loss=0.8045  val_macro_f1=0.3099  val_micro_f1=0.3357  val_auc_pr=0.3010  (15.9s)  *
  epoch   4

## 5. GTZAN — Task 2 genre classification

The spec asks for genre classification on GTZAN or FMA-small. Tasks 1/3/4 are *not* run here: GTZAN's only text is its genre label, so the text branch trivially reproduces the target.

In [10]:
!python src/preprocess.py --config configs/gtzan.yaml --export-samples 20
!python src/train.py baselines --config configs/gtzan.yaml --set device=cuda --set train.epochs=30 --set train.batch_size=64
!python src/train.py task2 --config configs/gtzan.yaml --set device=cuda --set train.epochs=30 --set train.batch_size=64
!python src/train.py task2 --config configs/gtzan.yaml --set device=cuda --set gnn.conv=gat --set train.epochs=30 --set train.batch_size=64

source=gtzan  graph=segment  text=distilbert-base-uncased
  gtzan: 1000 records on disk
  featurised 25 tracks...
  featurised 50 tracks...
  featurised 75 tracks...
  featurised 100 tracks...
  featurised 125 tracks...
  featurised 150 tracks...
  featurised 175 tracks...
  featurised 200 tracks...
  featurised 225 tracks...
  featurised 250 tracks...
  featurised 275 tracks...
  featurised 300 tracks...
  featurised 325 tracks...
  featurised 350 tracks...
  featurised 375 tracks...
  featurised 400 tracks...
  featurised 425 tracks...
  featurised 450 tracks...
  featurised 475 tracks...
  featurised 500 tracks...
  featurised 525 tracks...
  featurised 550 tracks...
/content/project/src/audio_features.py:34: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(str(path), sr=sample_rate, mono=True, duration=duration)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of libr

### Graph construction comparison

The spec calls out chord-transition graphs specifically. Each `graph.kind` needs its own preprocessing, so each writes to its own directory.

In [ ]:
# subprocess rather than `!` -- a shell escape inside a Python loop with line
# continuations is the kind of thing that silently runs the wrong command.
import subprocess, sys

for kind in ["segment", "chord", "hybrid"]:
    for cmd in (
        ["python", "src/preprocess.py", "--config", "configs/gtzan.yaml",
         "--set", f"graph.kind={kind}",
         "--set", f"paths.processed=data/processed_gtzan_{kind}",
         "--set", f"paths.splits=data/splits_gtzan_{kind}",
         "--export-samples", "0"],
        ["python", "src/train.py", "task2", "--config", "configs/gtzan.yaml",
         "--set", "device=cuda", "--set", f"graph.kind={kind}",
         "--set", f"paths.processed=data/processed_gtzan_{kind}",
         "--set", f"paths.splits=data/splits_gtzan_{kind}",
         "--set", f"paths.results=results_gtzan_{kind}",
         "--set", "train.epochs=30", "--set", "train.batch_size=64"],
    ):
        print(">", " ".join(cmd))
        r = subprocess.run(cmd)
        if r.returncode:
            print(f"  FAILED ({kind}) -- continuing with the other graph kinds")
            break

> python src/preprocess.py --config configs/gtzan.yaml --set graph.kind=segment --set paths.processed=data/processed_gtzan_segment --set paths.splits=data/splits_gtzan_segment --export-samples 0
> python src/train.py task2 --config configs/gtzan.yaml --set device=cuda --set graph.kind=segment --set paths.processed=data/processed_gtzan_segment --set paths.splits=data/splits_gtzan_segment --set paths.results=results_gtzan_segment --set train.epochs=30 --set train.batch_size=64
> python src/preprocess.py --config configs/gtzan.yaml --set graph.kind=chord --set paths.processed=data/processed_gtzan_chord --set paths.splits=data/splits_gtzan_chord --export-samples 0


## 6. DEAM — the auxiliary emotion term

The spec's multi-task loss is
`L = L_tags + α‖v − v̂‖² + β‖a − â‖²`, with valence/arousal from DEAM. DEAM
carries no tags, so read the MAE/R² here, not the F1.

In [ ]:
!python tools/fetch_data.py deam --extract
!python src/preprocess.py --config configs/deam.yaml --export-samples 20
!python src/train.py task3 --config configs/deam.yaml --mode cross_attention \
    --set device=cuda --set train.epochs=30 --set train.batch_size=64

## 7. MusicCaps — Task 4's own corpus

MusicCaps ships captions and YouTube ids, not audio (it is not
redistributable), so clips are pulled individually with `yt-dlp`. Colab already
has `ffmpeg`, which the local Windows machine does not — this is the right place
to try it.

Expect losses: some videos are gone, and YouTube rate-limits. The fetcher keeps
a resume manifest, reports its success rate, and stops early if the rate
collapses. **If fewer than ~200 clips land, run Task 4 on MagnaTagATune instead
and say so in the report** — a documented deviation costs far less than a table
built on 40 clips.

In [ ]:
!pip install -q yt-dlp
!python tools/fetch_data.py musiccaps --clips 1200

In [ ]:
import glob, subprocess

n = len(glob.glob('data/raw/musiccaps/audio/*.wav'))
print(f"{n} MusicCaps clips on disk")

# MusicCaps clips are only 10 s, so segments must be ~1 s to give a graph at
# all -- at 2.5 s a clip is four nodes, which is not a graph worth message
# passing over.
COMMON = ["--set", "dataset.source=musiccaps",
          "--set", "paths.processed=data/processed_musiccaps",
          "--set", "paths.splits=data/splits_musiccaps",
          "--set", "audio.segment_seconds=1.0"]

if n >= 200:
    for cmd in (
        ["python", "src/preprocess.py", *COMMON,
         "--set", "dataset.clip_seconds=10",
         "--set", "dataset.min_tag_count=20",
         "--set", "dataset.text_field=caption",
         "--set", "dataset.text_tag_fraction=0.0",
         "--export-samples", "20"],
        ["python", "src/train.py", "task4", *COMMON,
         "--set", "paths.results=results_musiccaps",
         "--set", "device=cuda", "--set", "train.epochs=40",
         "--set", "train.batch_size=64"],
    ):
        print(">", " ".join(cmd))
        if subprocess.run(cmd).returncode:
            print("  FAILED -- fall back to the MagnaTagATune Task 4 result")
            break
else:
    print("\nToo few clips for a meaningful retrieval split. Use the "
          "MagnaTagATune Task 4\nresult and state the deviation in the "
          "report's Limitations (there is a \\FILL\nmarker for exactly this).")

## 8. Build the Task 4 listening study

Exports the retrieved clips and writes one self-contained HTML file. Send it to
at least 5 listeners (the spec's minimum), collect the `ratings_*.json` files
they download, and aggregate.

In [ ]:
# Prefer MusicCaps (real captions); else the textless-dropped MTAT retrieval run.
import os

CANDIDATES = [
    ("results_musiccaps", "data/processed_musiccaps"),
    ("results_mtat_r4",   "data/processed_mtat_r4"),
    ("results_mtat",      "data/processed_mtat"),
]
RES, PROC = next(((r, p) for r, p in CANDIDATES
                  if os.path.exists(f"{r}/retrieval_examples")), (None, None))
assert RES, "no Task 4 retrieval output found -- run a task4 cell first"
print("building study from", RES)

!python tools/make_human_eval.py --results $RES --processed $PROC --queries 10 --seconds 12

from google.colab import files
files.download(f"{RES}/human_eval/rating_form.html")

## 9. Tables and plots

In [ ]:
!python src/evaluate.py --set paths.results=results_mtat --set paths.plots=results_mtat/plots
!python src/evaluate.py --set paths.results=results_gtzan --set paths.plots=results_gtzan/plots

## 10. Download the results

Only `results_*` comes back — the graphs and raw audio stay here.

In [ ]:
!zip -qr results_gpu.zip results_mtat results_mtat_r4 results_gtzan results_gtzan_*     results_mtat_longsched results_deam results_musiccaps \
    data/processed_mtat/samples data/processed_gtzan/samples \
    data/processed_mtat/meta.json data/processed_mtat/label_space.json \
    data/processed_gtzan/meta.json data/processed_gtzan/label_space.json 2>/dev/null
!du -sh results_gpu.zip

from google.colab import files
files.download("results_gpu.zip")

## Back on your machine

```bash
unzip -o results_gpu.zip -d .
python src/evaluate.py --set paths.results=results_mtat --set paths.plots=results_mtat/plots
python tools/make_report_tables.py --results results_mtat
```

Then `report/main.tex` picks the numbers up automatically.